# DAG-SA v2 — run everything, unattended

**Runtime → Run all**, click through the Drive authorisation when it appears, then leave it.

It runs two campaigns on identical splits and seeds:

| Campaign | Units | What it answers |
|---|---|---|
| `ds2a_strong_v2` | 9 subjects × 8 seeds = 72 | Does embedding EEGNet and the exact B5 baseline as pool members help, on the dataset where EEGNet leads by 6.4 points? |
| `ds1_strong_v2` | 4 subjects × 8 seeds = 32 | The same question on Dataset 1 |

Four variants each: `V0_published` (reference), `V4u_enriched_unconstrained`, `V7u_strong_unconstrained`, `V7l_strong_locked` (a committee of the strong baselines fused by the searched operators).

**Expect roughly 4–7 hours** on a GPU runtime. You do not need to watch it.

### If it disconnects
Re-run the notebook. Every campaign resumes: finished `(subject, seed)` units are skipped, results are written to Drive after **every** unit, and a unit that crashes is logged and skipped rather than ending the run. Nothing is repeated and nothing is lost.

### What you get
`MyDrive/EEG_DAGSA/results/SUMMARY.md` — both campaigns, the accuracy table, the McNemar win/tie/loss column, and the diagnostic that says whether the search actually *selected* the strong members. Plus `run_all.log` with the full console output.

## 1. Setup — clone, install, mount Drive
The only interaction in the whole notebook is the Drive authorisation popup.

In [ ]:
REPO_URL = 'https://github.com/yazanjer/DAG-Ensembles-EEG.git'
BRANCH   = 'v2-improvements'
PROJECT_ROOT = '/content/drive/MyDrive/EEG_DAGSA'

import os, sys, glob, subprocess

CLONE_DIR = '/content/eeg_dagsa_repo'
if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'pull', 'origin', BRANCH])

hits = glob.glob(CLONE_DIR + '/**/run_all.py', recursive=True)
assert hits, f'run_all.py not found - is branch {BRANCH} pushed?'
CODE_DIR = os.path.dirname(hits[0])
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print('code dir:', CODE_DIR)

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'numpy', 'scipy', 'scikit-learn', 'pandas', 'matplotlib',
                'pyyaml', 'mne', 'pyriemann', 'tabulate'])
try:
    import torch
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch'])

from google.colab import drive
drive.mount('/content/drive')

DS1_DIR  = PROJECT_ROOT + '/dataset'
DS2A_DIR = PROJECT_ROOT + '/dataset_2a'

import numpy, sklearn, mne, pyriemann
print('deps:', numpy.__version__, sklearn.__version__, mne.__version__, pyriemann.__version__)
try:
    import torch; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
except Exception:
    print('torch NOT available - the EEGNet pool members will be skipped')

# fail now, loudly, rather than three hours in
for label, d, pattern in [('Dataset 1', DS1_DIR, 'BCICIV_calib_ds1?.mat'),
                          ('Dataset 2a', DS2A_DIR, 'A0?T.mat')]:
    found = glob.glob(os.path.join(d, pattern))
    assert found, f'{label}: no files matching {pattern} in {d}'
    print(f'{label}: {len(found)} files in {d}')

## 2. Run everything
This is the long cell — hours. Progress prints a line per unit with a live ETA, and the same output is appended to `results/run_all.log` on Drive, so you can close the tab and read it later.

To shorten the run, edit `CAMPAIGNS` below (fewer seeds is the cheapest cut; fewer subjects biases the result).

In [ ]:
import run_all

STRONG = ['V0_published', 'V4u_enriched_unconstrained',
          'V7u_strong_unconstrained', 'V7l_strong_locked']

CAMPAIGNS = [
    ('ds2a_strong_v2', 'ds2a', [1, 2, 3, 4, 5, 6, 7, 8, 9], list(range(42, 50)), STRONG),
    ('ds1_strong_v2',  'ds1',  ['a', 'b', 'f', 'g'],        list(range(42, 50)), STRONG),
]

summary_path = run_all.main(project_root=PROJECT_ROOT,
                            ds1_dir=DS1_DIR,
                            ds2a_dir=DS2A_DIR,
                            campaigns=CAMPAIGNS)

## 3. The result

Read the **W/T/L** column before the Δ column — a win or loss is only counted where McNemar reaches p < 0.05.

Then read the **strong (EEGNet / exact B5)** column. If it is near `0/72`, the search never selected a strong member and the accuracies say nothing about whether embedding them helps.

The noise floor measured on Dataset 1 is about **3 accuracy points** on a 32-unit mean, so treat anything smaller as a tie whichever way it points.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(summary_path).read()))